# Cleaning customer data using Python 

this project uses dataset found on git hub https://github.com/Riche-Lina/customer-data-cleaning/blob/main/README.md#-dataset-description

In [148]:
import pandas as pd
import numpy as np
import re
from dateutil.parser import parse
from datetime import datetime
import pycountry


In [149]:
customers = pd.read_csv('data/customers_dirty.csv')

customers.head()

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes
0,185.0,Michael,Johnson,michael.johnson@test.com,,Male,0000-00-00,06-13-23,GERMANY,LOS ANGELES,12545 Mueller Pike Suite 571,NaN,Reveal laugh attack and.
1,526.0,Daniel,Pacheco,daniel.pacheco@fakemail.xyz,+1-813-902-7050,NaN,0000-00-00,07/24/2023,Egypt,ALGIERS,31190 Harper Squares,No,NaN
2,40.0,Maurice,Barajas,maurice.barajas@gmail.com,,O,19-10-1976,21-03-2025,Algeria,MADRID,744 Hall Stravenue Apt. 518,True,NaN
3,114.0,Kirk,Morrison,,NaN,O,25-06-1967,06 Jan 2023,CA,Toronto,7059 Hill Turnpike Apt. 165,No,NaN
4,138.0,Janet,Vaughn,janet.vaughn276@hotmail.com,+33 2 20 95 54 79,O,"March 12, 1972",06-19-25,U.S.A.,Chigaco,8441 Huang Locks Suite 240,0,NaN


## Basic dataset information

shape shows us the dimension of the dataset (num rows, num cols)
info shows us more information about the dataset, column names, types and are there missing values

In [150]:
customers.shape

(538, 13)

In [151]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        535 non-null    float64
 1   first_name         535 non-null    str    
 2   last_name          535 non-null    str    
 3   email              507 non-null    str    
 4   phone              441 non-null    str    
 5   gender             444 non-null    str    
 6   birth_date         504 non-null    str    
 7   registration_date  535 non-null    str    
 8   country            535 non-null    str    
 9   city               535 non-null    str    
 10  address            528 non-null    str    
 11  is_active          451 non-null    str    
 12  notes              134 non-null    str    
dtypes: float64(1), str(12)
memory usage: 54.8 KB


### Initial check up

first we check up if there are any missing values in the dataset, then we check if there are any duplicated values

In [152]:
customers.isnull().sum()

customer_id            3
first_name             3
last_name              3
email                 31
phone                 97
gender                94
birth_date            34
registration_date      3
country                3
city                   3
address               10
is_active             87
notes                404
dtype: int64

### Checking up id col


Using info() we see that customer_id in float not integer.
And using isnull().sum() we see that 3 customer_ids are missing and we are going to remove those rows.


In [153]:
customers['customer_id'] = customers['customer_id'].astype('Int64')

In [154]:
customers.dropna(subset=['customer_id'], how='all', inplace=True)

customers.isnull().sum()

customer_id            0
first_name             0
last_name              0
email                 28
phone                 94
gender                91
birth_date            31
registration_date      0
country                0
city                   0
address                7
is_active             84
notes                401
dtype: int64

In [155]:
customers.duplicated().sum()

np.int64(0)

Becasue we removed null ids, we do not have any more duplicates in customer_id col but if we would have them we would use cell below to drop those cols

In [156]:
# removing all duplicate values
# customers.drop_duplicates(inplace = True) 
# inplace = True makes sure that the method doesn't return us new DataFrame, but that it changes the original one
# now checking for duplicates again
# customers.duplicated().sum()

In [157]:
id_summary = customers['customer_id'].describe()
print("--- Customer ID Statistical Summary ---")
print(id_summary)
print("\n----------------------------------------")

--- Customer ID Statistical Summary ---
count         535.0
mean          268.0
std      154.585467
min             1.0
25%           134.5
50%           268.0
75%           401.5
max           535.0
Name: customer_id, dtype: Float64

----------------------------------------


In [158]:
is_numerical = pd.api.types.is_numeric_dtype(customers["customer_id"])
print(f"Is numerical: {is_numerical}")

Is numerical: True


In [159]:
is_unique = customers["customer_id"].is_unique
print(f"Is unique: {is_unique}")

Is unique: True


In [160]:
sorted_ids = customers["customer_id"].sort_values()

# function is generating ideal array where there are no skips (ids move up by one)
is_perfect_sequence = np.array_equal(sorted_ids, np.arange(sorted_ids.min(), sorted_ids.max() + 1)) 

print(f"Increments strictly by 1: {is_perfect_sequence}")

Increments strictly by 1: True


In [161]:
mean_val = customers['customer_id'].mean()
median_val = customers['customer_id'].median()
mean_median_diff = abs(mean_val - median_val)

print(f"Mean (Average ID): {mean_val}")
print(f"Median (50th Percentile ID): {median_val}")
print(f"Distance between Mean and Median: {mean_median_diff}")

Mean (Average ID): 268.0
Median (50th Percentile ID): 268.0
Distance between Mean and Median: 0.0


In [162]:
print(customers['customer_id'].dtype) 

Int64


In [163]:
customers = customers.sort_values(by="customer_id").reset_index(drop=True)

In [164]:
customers.head()

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes
0,1,David,Alvarez,david.alvarez@company.co,00213694151326,O,03-10-1976,29/07/2025,United States of America,MADRID,963 Jennifer Locks Suite 787,Yes,NaN
1,2,David,walker,david.walker747@notarealdomain.zzz,476-4598,female,19601116,27.05.2023,GERMANY,PARIS,301 Jeremy Bypass,Yes,NaN
2,3,Michele,Jones,michele.jones@test.com,+213 528566572,NaN,"May 15, 1953",20260223,france,CHICAGO,82997 Kristen Valley Suite 311,Yes,NaN
3,4,Donna,Gomez,donna.gomez@fakemail.xyz,(500) 645-3591,X,09/05/1966,2023/08/23,United States of America,Algeirs,106 Mcbride Coves,N,unknown
4,5,John,Lee,john.lee372@mail.ru,2248786714,female,03-02-69,10/25/2024,united states,Berlin,47317 Brent Plains Apt. 132,No,NaN


## Cleaning Wrong Format

In [165]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   customer_id        535 non-null    Int64
 1   first_name         535 non-null    str  
 2   last_name          535 non-null    str  
 3   email              507 non-null    str  
 4   phone              441 non-null    str  
 5   gender             444 non-null    str  
 6   birth_date         504 non-null    str  
 7   registration_date  535 non-null    str  
 8   country            535 non-null    str  
 9   city               535 non-null    str  
 10  address            528 non-null    str  
 11  is_active          451 non-null    str  
 12  notes              134 non-null    str  
dtypes: Int64(1), str(12)
memory usage: 55.0 KB


Here we see that birth_date and registration_date are represented as strings not dates.

We want to convert is_active to boolean.

Gender to category.

In [166]:
# replacing invalid values with NA
for col in ["birth_date", "registration_date"]:
    customers[col] = customers[col].replace(
        ["0000-00-00", ""], pd.NA
    )

# Parse mixed date formats
customers["birth_date"] = pd.to_datetime(
    customers["birth_date"],
    format="mixed",
    errors="coerce", # if it can't be converted than it becomes NaN
    dayfirst=True
)

customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

In [167]:
print(customers[["birth_date", "registration_date"]].dtypes)

birth_date           datetime64[us]
registration_date    datetime64[us]
dtype: object


In [168]:
print(customers["is_active"].unique())

<StringArray>
['Yes', 'N', 'No', nan, 'Active', 'True', 'False', 'Y', 'Inactive', '0', '1']
Length: 11, dtype: str


In [169]:
customers["is_active"] = (
    customers["is_active"]
    .astype("string")
    .str.strip()
    .str.lower()
)

customers["is_active"] = customers["is_active"].map({
    "yes": True,
    "y": True,
    "true": True,
    "1": True,
    "active": True,
    "no": False,
    "n": False,
    "false": False,
    "0": False,
    "inactive": False
}).astype("boolean")

In [170]:
customers["gender"] = (
    customers["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
)

gender_mapping = {
    "f": "female",
    "female": "female",
    "m": "male",
    "male": "male",
    "o": "other",
    "other": "other",
    "unknown": "unknown",
    "x": "unknown"
}

customers["gender"] = (
    customers["gender"]
    .map(gender_mapping)
    .astype("category")
)

In [171]:
print(sorted(customers["gender"].dropna().unique()))

['female', 'male', 'other', 'unknown']


In [172]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        535 non-null    Int64         
 1   first_name         535 non-null    str           
 2   last_name          535 non-null    str           
 3   email              507 non-null    str           
 4   phone              441 non-null    str           
 5   gender             444 non-null    category      
 6   birth_date         464 non-null    datetime64[us]
 7   registration_date  535 non-null    datetime64[us]
 8   country            535 non-null    str           
 9   city               535 non-null    str           
 10  address            528 non-null    str           
 11  is_active          451 non-null    boolean       
 12  notes              134 non-null    str           
dtypes: Int64(1), boolean(1), category(1), datetime64[us](2), str(8)
memory usage

## Checking up string cols

### first_name and last_name

In [173]:
customers["first_name"].head()

0      David
1      David
2    Michele
3      Donna
4       John
Name: first_name, dtype: str

In [174]:
customers["last_name"].head()

0      Alvarez
1    walker   
2        Jones
3        Gomez
4          Lee
Name: last_name, dtype: str

In [175]:
customers["first_name"] = customers["first_name"].str.strip().str.title()

In [176]:
customers["last_name"] = customers["last_name"].str.strip().str.title()

### Country and City

In [177]:
print(sorted(customers["country"].dropna().unique()))

['ALGERIA', 'Algeria', 'Algerie', 'Alge\u200bria', 'Britain', 'CA', 'CANADA', 'Canada', 'Canad\ufeffa', 'Canda', 'DZ', 'Deutschland', 'EGYPT', 'Egpyt', 'Egypt', 'Egyp\ufefft', 'Eg\u200bpyt', 'Emirates', 'England', 'England\xa0', 'Espana', 'FRANCE', 'Fr', 'Franc', 'France', 'GERMANY', 'Germany', 'Germny', 'Ger\xa0many', 'G\ufeffermny', 'SPAIN', 'Spain', 'Spain ', 'Spa\ufeffin ', 'S\u200bpain ', 'U.A.E.', 'U.K.', 'U.S.A.', 'UAE', 'UK', 'US', 'USA', 'United Arab Emirates', 'United Kingdom', 'United States', 'United States of America', 'Usa', 'U\u200b.A.E.', 'algeria', 'canada', 'egypt', 'france', 'germany', 'spain', 'united kingdom', 'united states', '\xa0Egypt', '\xa0Franc', '\xa0egypt', '\ufeffAlgerie', '\ufeffEgypt']


In [178]:
customers["country"] = (
    customers["country"]
    .astype("string")
    .str.replace(r"[\u200b\u200e\u200f\ufeff\u00a0]", "", regex=True)
    .str.strip()
    .str.title()
)

In [179]:
country_mapping = {
    # Algeria
    "Algeria": "Algeria",
    "Dz": "Algeria",
    "Algerie": "Algeria",

    # Canada
    "Ca": "Canada",
    "Canada": "Canada",
    "Canda": "Canada",

    # Egypt
    "Egypt": "Egypt",
    "Egpyt": "Egypt",

    # France
    "Fr": "France",
    "Franc": "France",
    "France": "France",

    # Germany
    "Germany": "Germany",
    "Germny": "Germany",
    "Deutschland": "Germany",

    # Spain
    "Spain": "Spain",
    "Espana": "Spain",

    # United Arab Emirates
    "Uae": "United Arab Emirates",
    "U.A.E.": "United Arab Emirates",
    "Emirates": "United Arab Emirates",
    "United Arab Emirates": "United Arab Emirates",

    # United Kingdom
    "Uk": "United Kingdom",
    "U.K.": "United Kingdom",
    "Britain": "United Kingdom",
    "England": "United Kingdom",
    "United Kingdom": "United Kingdom",

    # United States
    "Us": "United States",
    "Usa": "United States",
    "U.S.A.": "United States",
    "United States": "United States",
    "United States Of America": "United States"
}

customers["country"] = customers["country"].replace(country_mapping)

In [180]:
print(sorted(customers["country"].dropna().unique()))

['Algeria', 'Canada', 'Egypt', 'France', 'Germany', 'Spain', 'United Arab Emirates', 'United Kingdom', 'United States']


In [181]:
customers["city"] = customers["city"].str.strip().str.title()

In [182]:
print(sorted(customers["city"].dropna().unique()))

['Algeirs', 'Alger', 'Algiers', 'Algies', 'Berlin', 'Berlinn', 'Bosotn', 'Boston', 'Cairo', 'Cairoo', 'Chicago', 'Chigaco', 'Duabi', 'Dubai', 'Houston', 'Huston', 'Kairo', 'L.A.', 'La', 'London', 'Londonn', 'Londres', 'Los Angelas', 'Los Angeles', 'Madird', 'Madrid', 'Maimi', 'Miami', 'New  York', 'New York', 'Newyork', 'Ney York', 'Paris', 'Pariss', 'Seatle', 'Seattle', 'Toronto', 'Torronto']


In [183]:
city_mapping = {
    # Algiers
    "Algeirs": "Algiers",
    "Alger": "Algiers",
    "Algies": "Algiers",

    # Berlin
    "Berlinn": "Berlin",

    # Boston
    "Bosotn": "Boston",

    # Cairo
    "Cairoo": "Cairo",
    "Kairo": "Cairo",

    # Chicago
    "Chigaco": "Chicago",

    # Dubai
    "Duabi": "Dubai",

    # Houston
    "Huston": "Houston",

    # Los Angeles
    "L.A.": "Los Angeles",
    "La": "Los Angeles",
    "Los Angelas": "Los Angeles",

    # London
    "Londonn": "London",
    "Londres": "London",

    # Madrid
    "Madird": "Madrid",

    # Miami
    "Maimi": "Miami",

    # New York
    "New  York": "New York",
    "Newyork": "New York",
    "Ney York": "New York",

    # Paris
    "Pariss": "Paris",

    # Seattle
    "Seatle": "Seattle",

    # Toronto
    "Torronto": "Toronto"
}

customers["city"] = (
    customers["city"]
    .str.strip()
    .str.title()
    .replace(city_mapping)
)

In [184]:
print(sorted(customers["city"].dropna().unique()))

['Algiers', 'Berlin', 'Boston', 'Cairo', 'Chicago', 'Dubai', 'Houston', 'London', 'Los Angeles', 'Madrid', 'Miami', 'New York', 'Paris', 'Seattle', 'Toronto']


#### checking for duplicates

In [185]:
duplicate_mask = customers.duplicated(
    subset=["first_name", "last_name", "birth_date", "city"],
    keep=False
)

possible_duplicates = customers.loc[duplicate_mask].sort_values(
    ["last_name", "first_name", "birth_date", "city"]
)

print(
    possible_duplicates[
        ["customer_id", "first_name", "last_name",
         "birth_date", "city", "email"]
    ].to_string(index=False)
)

 customer_id first_name last_name birth_date     city                             email
         169   Nicholas   Alvarez 1983-04-12  Houston         nicholas.alvarez@test.com
         512   Nicholas   Alvarez 1983-04-12  Houston         nicholas.alvarez@test.com
         480     Darryl  Anderson 1936-10-28    Paris         darryl.anderson@yahoo.com
         505     Darryl  Anderson 1936-10-28    Paris         darryl.anderson@yahoo.com
          67    Gregory  Anderson 1975-12-23   Madrid      gregory.anderson141@test.com
         514    Gregory  Anderson 1975-12-23   Madrid      gregory.anderson141@test.com
         120     Tanner      Bird 1945-02-04   Berlin        tanner.bird550@hotmail.com
         513     Tanner      Bird 1945-02-04   Berlin        tanner.bird550@hotmail.com
          56       Mary Carpenter 1985-06-01   Madrid          mary.carpenter@yahoo.com
         528       Mary Carpenter 1985-06-01   Madrid          mary.carpenter@yahoo.com
         305     Ariana      Dia

In [186]:
duplicate_columns = [
    "first_name",
    "last_name",
    "birth_date",
    "city",
    "email"
]

# Count rows before cleaning
before = len(customers)

# Remove exact duplicates based on these columns
customers = customers.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).reset_index(drop=True)

print("Rows removed:", before - len(customers))
print("Rows remaining:", len(customers))

Rows removed: 33
Rows remaining: 502


### Phone-Number

In [187]:
customers["phone"] = (
    customers["phone"]
    .astype("string")
    .str.strip()
)

print(customers["phone"].head(20))

0        00213694151326
1              476-4598
2        +213 528566572
3        (500) 645-3591
4            2248786714
5         call me maybe
6          940.784.8744
7                  <NA>
8         call me maybe
9                      
10                 <NA>
11       00213613080933
12    +33 3 79 14 57 84
13        call me maybe
14       +213 778371272
15                 <NA>
16             880-7211
17        call me maybe
18       (997) 635-4585
19           9902150925
Name: phone, dtype: string


In [188]:
print(customers["phone"].isna().sum())

87


In [189]:
has_letters = customers["phone"].str.contains(
    r"[A-Za-z]",
    na=False
)

print(customers.loc[has_letters, ["customer_id", "phone"]])

     customer_id          phone
5              6  call me maybe
8              9  call me maybe
13            14  call me maybe
17            18  call me maybe
22            23  call me maybe
36            37  call me maybe
80            81  call me maybe
83            84  call me maybe
89            90  call me maybe
94            95  call me maybe
98            99  call me maybe
100          101  call me maybe
112          113  call me maybe
132          133  call me maybe
142          143  call me maybe
143          144  call me maybe
154          155  call me maybe
156          157  call me maybe
162          163  call me maybe
166          167  call me maybe
168          169  call me maybe
169          170  call me maybe
171          172  call me maybe
176          177  call me maybe
178          179  call me maybe
185          186  call me maybe
223          224  call me maybe
241          242  call me maybe
258          259  call me maybe
259          260  call me maybe
261     

In [190]:
mask = customers["phone"].str.lower().eq("call me maybe")

print("Invalid phone values:", mask.sum())
# mask checks every val in phone col and chekcs is it equal to call me maybe 

Invalid phone values: 47


In [191]:
customers.loc[mask, "phone"] = pd.NA 
# loc[] -> selects rows by their index labels


In [192]:
print(customers["phone"].isna().sum())
print(customers["phone"].head(20))

134
0        00213694151326
1              476-4598
2        +213 528566572
3        (500) 645-3591
4            2248786714
5                  <NA>
6          940.784.8744
7                  <NA>
8                  <NA>
9                      
10                 <NA>
11       00213613080933
12    +33 3 79 14 57 84
13                 <NA>
14       +213 778371272
15                 <NA>
16             880-7211
17                 <NA>
18       (997) 635-4585
19           9902150925
Name: phone, dtype: string


In [193]:
customers["phone"] = (
    customers["phone"]
    .str.strip()
    .str.replace(r"[().\-\s]", "", regex=True)
)

In [194]:
customers["phone"] = customers["phone"].str.replace(
    r"^00", "+", regex=True
)

In [195]:
print(customers["phone"].head(20))

0     +213694151326
1           4764598
2     +213528566572
3        5006453591
4        2248786714
5              <NA>
6        9407848744
7              <NA>
8              <NA>
9                  
10             <NA>
11    +213613080933
12     +33379145784
13             <NA>
14    +213778371272
15             <NA>
16          8807211
17             <NA>
18       9976354585
19       9902150925
Name: phone, dtype: string


In [196]:
import phonenumbers

def check_phone(phone):
    if pd.isna(phone) or not phone.strip():
        return pd.NA

    try:
        number = phonenumbers.parse(phone, None)
        return phonenumbers.is_valid_number(number)
    except phonenumbers.NumberParseException:
        return False

customers["phone_valid"] = customers["phone"].apply(check_phone)

In [197]:
print(
    customers.loc[
        customers["phone_valid"] == False,
        ["customer_id", "country", "phone"]
    ].head(20)
)

    customer_id               country          phone
1             2               Germany        4764598
2             3                France  +213528566572
3             4         United States     5006453591
4             5         United States     2248786714
6             7         United States     9407848744
16           17  United Arab Emirates        8807211
18           19               Germany     9976354585
19           20  United Arab Emirates     9902150925
20           21               Germany        2634388
23           24        United Kingdom        9324249
25           26               Germany  +213575437844
28           29        United Kingdom  +213756478906
30           31               Algeria        3179922
31           32                 Egypt   +12934341339
32           33                Canada  +213714991164
37           38               Algeria     4281341978
40           41         United States     4053359837
42           43               Germany   +12385

### Email

In [198]:
print(customers["email"].head(20))
print("Missing emails:", customers["email"].isna().sum())
print("Unique emails:", customers["email"].nunique())

0                   david.alvarez@company.co
1       david.walker747@notarealdomain.zzz  
2                     michele.jones@test.com
3                 donna.gomez@fakemail.xyz  
4                        john.lee372@mail.ru
5                     richard.olson@test.com
6                 wendy.peters686@company.co
7            samantha.schmidt474@yahoo.com  
8                  melanie.bell@fakemail.xyz
9                   david.brewer@outlook.com
10                                          
11                           julia.moore567@
12                    brent.waltersgmail.com
13                  charles.ware755@test.com
14                    rachel.flynn72@mail.ru
15               harry.middleton46@gmail.com
16                michael.hoover@outlook.com
17                ricardo.zavala@hotmail.com
18                                       NaN
19                    amanda.ball883@mail.ru
Name: email, dtype: str
Missing emails: 26
Unique emails: 449


In [199]:
customers.shape

(502, 14)

In [200]:
535 - 28

507

In [201]:
customers["email"] = (
    customers["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [202]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

customers["email_valid"] = customers["email"].str.fullmatch(
    email_pattern,
    na=False
)

invalid_emails = customers.loc[
    ~customers["email_valid"],
    ["customer_id", "email"]
]

customers.loc[~customers["email_valid"], "email"] = pd.NA

In [203]:
duplicate_emails = customers[
    customers["email"].notna()
    & customers["email"].duplicated(keep=False)
].sort_values("email")

print(duplicate_emails[["customer_id", "email"]])

     customer_id                            email
73            74       aaron.garrison@outlook.com
303          304       aaron.garrison@outlook.com
202          203   aaron.woods@notarealdomain.zzz
397          398   aaron.woods@notarealdomain.zzz
157          158  angela.williams504@fakemail.xyz
70            71  angela.williams504@fakemail.xyz
241          242     april.hernandez@fakemail.xyz
131          132     april.hernandez@fakemail.xyz
121          122  april.torres@notarealdomain.zzz
455          456  april.torres@notarealdomain.zzz
304          305          ariana.diaz@outlook.com
301          302          ariana.diaz@outlook.com
226          227  ashley.lopez@notarealdomain.zzz
107          108  ashley.lopez@notarealdomain.zzz
486          487         caroline.collins@mail.ru
487          488         caroline.collins@mail.ru
272          273         daniel.ross708@yahoo.com
212          213         daniel.ross708@yahoo.com
159          160       jennifer.poole@outlook.com


In [205]:
shared_emails = customers[
    customers["email"].notna()
    & customers["email"].duplicated(keep=False)
].sort_values("email")

print(
    shared_emails[
        ["email", "customer_id", "first_name", "last_name"]
    ].to_string(index=False)
)

                          email  customer_id first_name last_name
     aaron.garrison@outlook.com           74      Aaron  Garrison
     aaron.garrison@outlook.com          304      Emily     Young
 aaron.woods@notarealdomain.zzz          203      Aaron     Woods
 aaron.woods@notarealdomain.zzz          398     Leslie       Fox
angela.williams504@fakemail.xyz          158     Angela  Williams
angela.williams504@fakemail.xyz           71      Ellen      Paul
   april.hernandez@fakemail.xyz          242       Adam    Miller
   april.hernandez@fakemail.xyz          132      April Hernandez
april.torres@notarealdomain.zzz          122      April    Torres
april.torres@notarealdomain.zzz          456      Kayla   Johnson
        ariana.diaz@outlook.com          305     Ariana      Diaz
        ariana.diaz@outlook.com          302      Jason Whitehead
ashley.lopez@notarealdomain.zzz          227    Felicia  Thompson
ashley.lopez@notarealdomain.zzz          108     Ashley     Lopez
       car

In [210]:
customers.head()

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,phone_valid,email_valid
0,1,David,Alvarez,david.alvarez@company.co,+213694151326,other,1976-10-03,2025-07-29,United States,Madrid,963 Jennifer Locks Suite 787,True,NaN,True,True
1,2,David,Walker,david.walker747@notarealdomain.zzz,4764598,female,1960-11-16,2023-05-27,Germany,Paris,301 Jeremy Bypass,True,NaN,False,True
2,3,Michele,Jones,michele.jones@test.com,+213528566572,NaN,1953-05-15,2026-02-23,France,Chicago,82997 Kristen Valley Suite 311,True,NaN,False,True
3,4,Donna,Gomez,donna.gomez@fakemail.xyz,5006453591,unknown,1966-05-09,2023-08-23,United States,Algiers,106 Mcbride Coves,False,unknown,False,True
4,5,John,Lee,john.lee372@mail.ru,2248786714,female,2069-02-03,2024-10-25,United States,Berlin,47317 Brent Plains Apt. 132,False,NaN,False,True


## Checking 

Sicne the data was found on the gitHub now we are going to compare mine cleaned data to cleaned data from gitHub so that I can get some insights

In [208]:
reference = pd.read_csv("data/customers_clean.csv")

In [209]:
print("Mine cleaned data:", customers.shape)
print("Reference data:", reference.shape)

print("Mine columns:", customers.columns.tolist())
print("Reference columns:", reference.columns.tolist())

Mine cleaned data: (502, 15)
Reference data: (535, 15)
Mine columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'birth_date', 'registration_date', 'country', 'city', 'address', 'is_active', 'notes', 'phone_valid', 'email_valid']
Reference columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'birth_date', 'registration_date', 'country', 'city', 'address', 'is_active', 'notes', 'customer_id_outlier', 'email_valid']


In [211]:
print("Mine missing values:")
print(customers.isna().sum())

print("\nReference missing values:")
print(reference.isna().sum())

Mine missing values:
customer_id            0
first_name             0
last_name              0
email                 84
phone                134
gender                82
birth_date            66
registration_date      0
country                0
city                   0
address                6
is_active             82
notes                374
phone_valid          169
email_valid            0
dtype: int64

Reference missing values:
customer_id              0
first_name               0
last_name                0
email                   88
phone                  219
gender                  91
birth_date             120
registration_date        1
country                  0
city                     0
address                  7
is_active               84
notes                  490
customer_id_outlier      0
email_valid              0
dtype: int64
